# UBS Colab Worker Node (Browser)\nThis notebook is the **Worker Node**. It runs the browser and connects out to Cloud Run signaling server.

In [ ]:
!pip install -q playwright flask requests pyngrok
!playwright install chromium

In [ ]:
import os, requests, threading, time, uuid
from flask import Flask, request, jsonify
from playwright.sync_api import sync_playwright
from pyngrok import ngrok

SIGNAL_URL = os.getenv('UBS_SIGNAL_URL', 'https://YOUR-CLOUD-RUN-URL')
REGION = os.getenv('UBS_REGION', 'us')
NODE_ID = os.getenv('UBS_NODE_ID', 'colab-' + str(uuid.uuid4())[:8])

In [ ]:
app = Flask(__name__)

@app.route('/run', methods=['POST'])
def run():
    data = request.get_json() or {}
    url = data.get('url')
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url, wait_until='domcontentloaded')
        title = page.title()
        html = page.content()[:5000]
        browser.close()
        return jsonify({'ok': True, 'title': title, 'htmlSnippet': html})

threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000), daemon=True).start()

In [ ]:
public_url = ngrok.connect(5000, bind_tls=True).public_url
print('Worker connect URL:', public_url)

register_payload = {
  'id': NODE_ID,
  'connectUrl': public_url,
  'region': REGION,
  'capabilities': ['playwright']
}
r = requests.post(f'{SIGNAL_URL}/signal/register', json=register_payload, timeout=20)
print('Register status:', r.status_code, r.text)

In [ ]:
try:
  while True:
    hb = requests.post(f'{SIGNAL_URL}/signal/heartbeat', json={'id': NODE_ID}, timeout=20)
    print('heartbeat', hb.status_code)
    time.sleep(240)
except KeyboardInterrupt:
  print('Stopping worker node heartbeat')

## ✅ SUCCESS\nWorker node is live and connected to signaling server. You can close this tab.